# 07 - Feature Matrix Builder
Build the game-level feature matrix using FeatureBuilder and save to Parquet.
Requires: All Phase 1 caches populated (notebooks 01-05) + notebook 06 (per-game logs).
Output: data/features/feature_matrix.parquet

**Note on SP recent form:** FeatureBuilder uses `pitching_stats_range()` to compute
30-day rolling ERA for each game date. On first run, this fetches and caches ~1,800
unique date-range queries from FanGraphs with 2-second rate limiting (~60 minutes).
Subsequent runs read from cache instantly. This is a one-time ingestion cost.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.features.feature_builder import FeatureBuilder
import pandas as pd
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
# Ensure INFO logging is visible in Jupyter to track stage-by-stage progress
import logging
logging.getLogger('src.features').setLevel(logging.INFO)

print("Starting FeatureBuilder.build() -- 10 pipeline stages:")
print("  1. Load schedule  2. Filter TBD starters  3. Outcome label")
print("  4. SP features  5. Offense features  6. Rolling features")
print("  7. Bullpen features  8. Park features")
print("  9. Advanced features  (SP recent form: ~60 min UNCACHED, instant if cached)")
print(" 10. Kalshi features")
print("\nWatch for 'Adding ... features...' log messages to confirm progress.")
print("If no output for > 2 min: SP recent form ingestion is running (~1800 API calls at 2s each).\n")

builder = FeatureBuilder(seasons=list(range(2015, 2025)))
df = builder.build()
print(f"\nDone! Feature matrix: {df.shape[0]} games x {df.shape[1]} columns")
print(f"Seasons: {sorted(df['season'].unique())}")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")

In [ ]:
print("Columns and dtypes:")
for col in sorted(df.columns):
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    print(f"  {col:30s}  {str(df[col].dtype):15s}  {non_null:6d} non-null ({pct:.1f}%)")

In [ ]:
output_dir = Path("../data/features")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "feature_matrix.parquet"
df.to_parquet(output_path, engine="pyarrow", compression="snappy", index=False)
print(f"Saved to {output_path} ({output_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
# Read back and verify
df_check = pd.read_parquet(output_path)
assert df_check.shape == df.shape, f"Shape mismatch: {df_check.shape} vs {df.shape}"
assert "home_win" in df_check.columns, "Missing home_win column"
assert "game_date" in df_check.columns, "Missing game_date column"
assert "season" in df_check.columns, "Missing season column"
assert "sp_recent_era_diff" in df_check.columns, "Missing sp_recent_era_diff (30-day pitching_stats_range ERA)"
print(f"Verification passed: {df_check.shape[0]} rows, {df_check.shape[1]} columns")

In [ ]:
print(f"\nHome win rate: {df['home_win'].mean():.3f}")
print(f"Games per season:")
print(df.groupby('season').size().to_string())
print(f"\nKalshi coverage: {df['kalshi_yes_price'].notna().sum()} games with Kalshi prices")
print(f"SP recent form coverage per season:")
sp_coverage = df.groupby('season')['sp_recent_era_diff'].apply(
    lambda x: f"{x.notna().sum():4d}/{len(x):4d} ({100*x.notna().mean():.1f}%)"
)
for season, cov in sp_coverage.items():
    print(f"  {season}: {cov}")